# Step 2-4. 내 팀의 라우팅을 증류하기
**Distillation(증류)** 은 큰 모델(스승)의 답을 작은 모델(제자)이 따라 하도록 가르치는 방법입니다. **Step 2-3의 팀**에서 지휘자가 하는 "전문가 고르기"를 작은 모델에게 가르칩니다. (한 단계씩 뜯어 보려면 Step 2-5, 2-6)

사진을 보는 일과 분석은 큰 모델에 그대로 두고, **답의 형태가 정해진 일(누구에게 보낼지)만** 작은 모델로 옮깁니다.

## 순서
1. 질문을 만들고 스승(큰 모델)에게 답을 달게 한다 (2\~3분)
2. 학습 전의 제자에게 시험을 보게 한다
3. 학습시킨다 (200번 반복, 2분쯤)
4. 학습 후 같은 시험을 보게 한다
5. 100번 학습한 모델과 200번 학습한 모델을 비교한다
6. 제자가 얼마나 작은지 숫자로 본다
7. 지휘자 자리에 넣는다

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1~2분 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "mlx-lm", "ollama", "strands-agents[ollama]", "numpy", "requests", "rdkit"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정
어느 팀을 증류할지 고릅니다. 처음에는 완성 예시인 `xray_team`으로 끝까지 해 보세요.

### 내 팀(`my_team`)으로 하려면
1. 아래 셀의 `TEAM_FILE`을 `"my_team"`으로 바꾸고, **이 셀부터 아래로 차례대로 다시 실행**합니다.
2. **학습은 처음부터 다시 합니다.** 팀이 다르면 전문가도 질문도 다르기 때문에, 엑스레이 팀으로 학습한 모델을 쓸 수 없습니다. 학습 자료와 모델은 팀마다 따로 저장됩니다(`distill/<팀 이름>/`). 엑스레이 팀의 것은 지워지지 않습니다.
3. 질문은 `my_team.py` 안의 `TRAIN_TEMPLATES`(학습용 질문 틀), `TEST_TEMPLATES`(시험용 질문 틀), `SUBJECTS`(질문 틀의 `{d}` 자리에 들어갈 낱말)로 만듭니다. 내 팀의 주제를 건강검진에서 다른 것으로 바꿨다면 이 세 가지도 내 주제에 맞게 고쳐야 합니다. Kiro에게 "my_team.py의 질문 틀을 내 팀 주제에 맞게 고쳐 줘"라고 시켜도 됩니다.
4. 아래쪽 8번의 **시험해 볼 질문**도 내 팀에 맞게 고칩니다.
5. `my_team.py`의 전문가나 질문 틀을 고쳤다면 그때마다 다시 학습해야 합니다. `REUSE = True`로 두었더라도 팀 파일이 달라졌으면 알아서 새로 만들고 다시 학습합니다.

In [ ]:
import importlib, os, sys, time, warnings

os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

TEAM_FILE = "xray_team"      # TODO: 내 팀으로 하려면 "my_team" (위의 설명을 먼저 읽으세요. 학습을 처음부터 다시 합니다)

REUSE = False
# REUSE = True     # 미리 만들어 둔 학습 자료와 학습해 둔 모델을 쓰려면 이 줄 맨 앞의 # 을 지우세요.
#                  # 스승에게 묻기(1~2분)와 학습(2분쯤)을 건너뜁니다. 처음 실행할 때는 그대로 둡니다
import agent_team, lab_ui
importlib.reload(lab_ui); importlib.reload(agent_team)      # 파일이 바뀌었으면 다시 읽습니다 (이 셀만 다시 실행해도 최신 코드가 됩니다)
module = importlib.reload(importlib.import_module(TEAM_FILE))
team = module.TEAM

from agent_team import EXPERT_MODEL, STUDENT_MODEL
print(f"✅ {team.goal}")
print(f"   스승: {EXPERT_MODEL}   제자: {STUDENT_MODEL}")
print(f"   제자에게 줄 지시문: {team.instruction!r}")

## 2. 질문 만들고 스승에게 답 달게 하기
팀 파일의 질문 틀(`TRAIN_TEMPLATES`)에 낱말(`SUBJECTS`)을 넣어 질문을 만듭니다. `TEST_TEMPLATES`는 학습에 쓰지 않고, **처음 보는 유형**의 시험에만 씁니다.

끝나면 조합별 개수가 나옵니다. 한 조합이 대부분이면 제자는 그 답만 외웁니다. 그럴 때는 질문 틀을 고르게 늘립니다.

In [ ]:
train_rows, test_rows = team.build_distill_data(module.TRAIN_TEMPLATES, module.TEST_TEMPLATES, module.SUBJECTS, reuse=REUSE)

import random
print("\n스승의 답 몇 개만 직접 보기 (스승이 틀리면 제자도 틀립니다):")
for r in random.Random(1).sample(train_rows, 8):
    print(f"   {r['experts']:<26} {r['question']}")

## 3. 학습 전 시험
제자가 스승과 **똑같은 전문가 조합**을 고르면 정답으로 칩니다.

In [ ]:
print("학습 전, 처음 보는 유형의 질문:")
before_test = team.exam("test", trained=False, show=6)
before_valid = team.exam("valid", trained=False)
print(f"\n📊 학습 전 정답률  본 유형 {before_valid:.0%}   처음 보는 유형 {before_test:.0%}")

## 4. 학습
200번 반복, 학습률 1e-4로 학습합니다. 2분쯤 걸리고 그동안 아이맥의 팬이 돌 수 있습니다. 진행 줄의 "틀린 정도(loss)"가 줄어드는지 보세요.

100번 반복할 때마다 그때의 모델을 따로 저장해 둡니다. 뒤에서 서로 비교합니다. `iters=600`으로 늘리면 600번까지 학습합니다(약 5분).

> 발표자 맥에서 600번까지 학습하며 100번마다 잰 "처음 보는 유형" 정답률은 78%, 92%, 60%, 96%, 84%, 88%였습니다. 오래 학습한다고 늘 좋아지지 않았고, 200번이면 충분했습니다.

In [ ]:
team.train_student(iters=200, learning_rate=1e-4, reuse=REUSE)

## 5. 학습 후 시험

In [ ]:
print("학습 후, 처음 보는 유형의 질문:")
after_test = team.exam("test", show=6)
after_valid = team.exam("valid")

print(f"\n📊 정답률 (스승과 같은 답을 고른 비율)")
print(f"                  학습 전    학습 후")
print(f"   본 유형          {before_valid:>5.0%}     {after_valid:>5.0%}")
print(f"   처음 보는 유형   {before_test:>5.0%}     {after_test:>5.0%}")

## 6. 학습을 얼마나 해야 할까: 100번마다 비교
학습 도중 100번마다 저장해 둔 모델에게 **같은 시험**을 보게 해서 표와 그래프로 비교합니다. 0번은 학습 전의 모델입니다. 모델 하나에 20초쯤 걸립니다.

- 어느 시점부터 점수가 더 오르지 않나요? 그 뒤의 학습은 시간만 쓴 것입니다.
- "본 유형"은 그대로인데 "처음 보는 유형"이 오르내린다면, 학습 자료의 말투에만 맞춰지고 있는 것입니다. 시험 문제가 50개뿐이라 몇 문제 차이로도 크게 흔들립니다.
- 600번까지 보고 싶으면 위의 학습 셀을 `iters=600`으로 바꿔 다시 실행한 뒤 이 셀을 실행합니다.

In [ ]:
rows = team.compare_checkpoints()

## 7. 제자는 얼마나 작은가
"작은 모델"이 얼마나 작은지 숫자로 봅니다. 방금 학습시킨 제자, 스승, 그리고 GPT나 Claude 같은 인터넷 너머의 모델을 한 표에 놓습니다. 제자의 메모리는 이 컴퓨터에서 실제로 돌려서 잽니다.

In [ ]:
sizes = team.show_sizes()

**이 표에서 볼 것**

- 제자는 스승보다 파라미터가 8분의 1, 파일은 12분의 1입니다. 메모리 1GB 안팎이면 돌아서, GPU 서버 없이 보통 노트북에서도 됩니다.
- GPT나 Claude는 크기를 공개하지 않고, 내려받아 내 컴퓨터에서 돌릴 수도 없습니다. 질문을 인터넷으로 그 회사의 서버에 보내고, 쓴 만큼 돈을 냅니다. 2026년 11월에 알려진 요금으로는 입력 100만 token당 GPT-6.1 Terra가 1달러, Claude Opus 5.5가 4달러입니다(요금은 바뀌므로 각 회사의 요금표를 확인하세요).
- **그렇다고 제자가 그 모델들만큼 똑똑한 것은 아닙니다.** 제자는 "전문가 고르기" 하나만 배웠고, 그 일에서 스승과 같은 답을 고르는 비율이 위에서 본 정도입니다. 글을 쓰거나 사진을 읽는 일은 못 합니다.
- 그래서 얻는 결론은 이것입니다. **답의 형태가 정해진 단순한 일(분류, 고르기, 정보 뽑기)까지 GPT나 Claude를 부를 필요는 없습니다.** 그런 일은 작은 모델을 직접 가르쳐 내 컴퓨터에서 돌릴 수 있고, 돈이 들지 않으며, 자료가 컴퓨터 밖으로 나가지 않습니다. 환자 자료처럼 밖으로 보내기 어려운 자료를 다룰 때 특히 쓸모가 있습니다.

## 8. 제자를 지휘자 자리에 넣기
`team.ask()`의 `router` 자리에 증류한 모델을 넘깁니다. 전문가를 고르는 일만 작은 모델이 하고, 사진을 보고 분석하는 일은 큰 모델이 그대로 합니다.

In [ ]:
route_with_student = team.load_student()

# 스승과 제자에게 같은 질문을 주고, 누구를 고르는지와 걸린 시간을 비교합니다.
# TODO: 질문을 바꿔 보세요! 학습 자료와 말투가 다른 질문일수록 스승과 다르게 고르는 일이 늘어납니다
if TEAM_FILE == "xray_team":
    questions = ["심장 그림자가 커 보이는지 봐 줘", "이런 사진 전에도 있었어?", "기흉 진료 지침의 근거를 찾아 줘"]
else:       # my_team: 건강검진 상담 팀 예시에 맞춘 질문입니다. 내 팀의 주제를 바꿨다면 이 질문도 내 주제에 맞게 고치세요
    questions = ["이 수치면 괜찮은 거야?", "운동은 어떻게 하면 좋을까?", "전부 확인해서 정리해 줘"]

team.route_with_teacher(questions[0])      # 스승 모델을 미리 메모리에 올려 둡니다
for q in questions:
    t0 = time.time(); a = team.route_with_teacher(q); t1 = time.time(); b = route_with_student(q); t2 = time.time()
    print(f"{q}\n   스승 {t1 - t0:.2f}초  {a}\n   제자 {t2 - t1:.2f}초  {b}   {'⭕ 같음' if a == b else '❌ 다름'}\n")

In [ ]:
# 1분쯤 걸립니다
case = team.cases[0] if team.cases else ""
if TEAM_FILE == "xray_team":
    question = "이 사진과 비슷한 과거 사례가 있어?"
else:       # my_team: 내 팀의 주제를 바꿨다면 이 질문도 고치세요
    question = "검사 수치 가운데 이상한 게 있어?"
result = team.ask(question, case=case, router=route_with_student)

## 9. 서비스에서도 쓰기
학습된 Adapter는 `distill/<팀 이름>/adapter`에 저장됩니다. 아래 셀로 Step 2-3의 웹 화면을 **다시 띄우면** 자동으로 찾아 씁니다. 브라우저가 저절로 열립니다.

In [ ]:
from lab_ui import start_service

start_service(TEAM_FILE)

웹 화면의 "전문가를 고르는 모델"에 **증류한 작은 모델**이 생깁니다. 큰 모델로 바꿔 가며 고르는 시간을 비교해 보세요.

## 정리
- 주제가 신약에서 엑스레이로 바뀌어도 증류의 순서는 같습니다: 질문 모으기 → 스승이 답 달기 → 학습 → 시험
- 증류한 것은 **라우팅뿐**입니다. 사진을 보는 일을 작은 모델에 옮기려면 사진을 다루는 학습 도구와, 무엇보다 **믿을 수 있는 정답**이 필요합니다. 스승이 사진을 틀리게 읽으면 제자는 그 틀린 답을 배웁니다
- 처음 보는 유형의 정답률이 낮으면 질문 틀이 부족한 것입니다. 실제 서비스에서는 사용자가 실제로 한 질문 기록을 씁니다
- 틀린 선택의 비용을 생각하세요. 이 팀은 알 수 없는 답이 나오면 전문가를 전부 부릅니다. 느려지더라도 빠뜨리지 않는 쪽입니다

## 더 해 보기
- `TEAM_FILE = "my_team"`으로 바꿔 내 팀을 증류해 보세요. 질문 틀이 적으면 정답률이 어떻게 되나요?
- 질문 틀을 10개 더 넣고 다시 학습하면 처음 보는 유형의 정답률이 오르나요?